# Fitting the CMB Angular Power Spectrum

AST 3414 - Spring 2026

## Introduction

In this lab, you'll use **Bayesian inference** and **Markov Chain Monte Carlo (MCMC)** to measure the Hubble constant from supernova data which we previously fit with $\chi^2$.

### What You'll Learn

1. Load and visualize the observed CMB temperature (TT) angular power spectrum.
2. Generate theoretical power spectra using the Boltzmann code **CAMB** for different cosmological parameters.
3. Construct a $\chi^2$ statistic comparing theory to data.
4. Use **MCMC** (with `emcee`) to estimate the matter density $\Omega_m$, dark energy density $\Omega_\Lambda$, and Hubble constant $H_0$.
5. Interpret your posterior distributions and compare to published Planck results.

---
## Setup

Run the cell below to install and then import the libraries we need.

In [ ]:
!pip install camb emcee corner healpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
import corner
import emcee
import camb
from scipy.optimize import minimize
import healpy as hp
# Plotting defaults
#rcParams['figure.figsize'] = (10, 6)
#rcParams['font.size'] = 14
#rcParams['figure.dpi'] = 100

print('All imports successful!')

---
## Part 1 — Load and Plot the Planck CMB Data

Cosmic Microwave Background is an (almost) uniform emission seen across the entire sky. Corresponding to a temperature of 2.726 K, it is a remnant of the earliest moments of the Big Bang. Number of missions, such as Planck, have imaged it.

This dataset corresponds to the temperature fluctuations from 2.726 K across the sky. Since the sky is spherical, and the dataset is somewhat specialized, special tools are used to read and interpret these data.
There is also a specialized function to take a power spectrum (square the magnitude of the Fourier transform) of this dataset.

In [ ]:
cmb_map = hp.read_map('https://irsa.ipac.caltech.edu/data/Planck/release_2/all-sky-maps/maps/component-maps/cmb/COM_CMB_IQU-commander_1024_R2.02_full.fits')
hp.mollview(cmb_map, min=-3e-4, max=3e-4, title="CMB only temperature map", unit="K")
plt.show()

The code below will generate the CMB angular power spectrum $\mathcal{D}_\ell \equiv \ell(\ell+1)\,C_\ell / 2\pi$ describing the variance of temperature fluctuations on the sky as a function of multipole moment $\ell$.

- **Low $\ell$** corresponds to large angular scales (the whole sky).  
- **High $\ell$** corresponds to small angular scales (fine details).

The power spectrum is computed from the FITS map using spherical harmonics transforms of the full sky using a function called ``anafast``. We do not need to remove the dipole or Galactic foreground because we are using a map where those have already been subtracted.

The positions and heights of the **acoustic peaks** are sensitive to cosmological parameters including $\Omega_m$, $\Omega_\Lambda$, and $H_0$.

In [ ]:
lmax = 3000
test_cls_meas_frommap = hp.anafast(cmb_map, lmax=lmax, use_pixel_weights=True)
ll = np.arange(lmax+1)
k2muK = 1e6
def multipole2ang(x):
    return 180./x
def ang2multipole(x):
    return 180./x
fig, ax = plt.subplots(layout='constrained')
plt.plot(ll, ll*(ll+1.)*test_cls_meas_frommap*k2muK**2/2./np.pi, '--', alpha=0.6, label='Planck 2018 PS from Data Map')
plt.xlabel(r'Multipole moment')
plt.ylabel(r'$D_\ell~[\mu K^2]$')
plt.xlim(.1,2000)
secax = ax.secondary_xaxis('top', functions=(multipole2ang, ang2multipole))
secax.set_xlabel(r'Angular scale (deg)')
secax.set_xticks(np.array([0.2,0.3,0.4,0.5,1,2,5]))

plt.grid()
plt.legend(loc='best')
plt.show()

### Questions

How many acoustic peaks can you identify? (You may wish to change to log scaling to better inspect the plot.) At roughly what multipole $\ell$ is the first (tallest) peak? What angular scale does that correspond to?

### Answer

---
## Part 2 — Theoretical Spectra with CAMB

Next we will use CAMB (Code for Anisotropies in the Microwave Background) to simulate a predicted model based on the cosmological parameters. The full dataset is too many data points, so below we have a simplified, binned version of the Planck 2018 power spectrum so that everything runs quickly.

In [ ]:
# ============================================================
# Planck 2018 binned TT power spectrum (representative bins)
# Columns: ell_center, D_ell [muK^2], sigma_D_ell [muK^2]
# Adapted from Planck 2018 results V. (arXiv:1907.12875)
# ============================================================
planck_data = np.array([
    #  ell,    D_ell,   sigma
    [   25,    850.0,    100.0],
    [   75,   1900.0,     55.0],
    [  125,   2700.0,     45.0],
    [  175,   3300.0,     42.0],
    [  200,   5600.0,     40.0],
    [  220,   5800.0,     38.0],
    [  275,   2800.0,     32.0],
    [  325,   1900.0,     29.0],
    [  375,   2700.0,     28.0],
    [  425,   3100.0,     27.0],
    [  475,   3500.0,     26.0],
    [  540,   3800.0,     27.0],
    [  600,   2400.0,     27.0],
    [  675,   1650.0,     28.0],
    [  750,   2400.0,     29.0],
    [  810,   2500.0,     30.0],
    [  870,   2350.0,     32.0],
    [  940,   1900.0,     34.0],
    [ 1020,   1500.0,     36.0],
    [ 1100,   2000.0,     40.0],
    [ 1200,   1750.0,     43.0],
    [ 1350,   1200.0,     50.0],
    [ 1500,   1050.0,     58.0],
    [ 1700,    750.0,     65.0],
    [ 2000,    500.0,     80.0],
])

ell_data = planck_data[:, 0]
Dl_data  = planck_data[:, 1]
Dl_sigma = planck_data[:, 2]

print(f'Loaded {len(ell_data)} data points, ell = {ell_data[0]:.0f} to {ell_data[-1]:.0f}')

We will use CAMB to vary just **three parameters** while holding the rest fixed at Planck best-fit values:

| Parameter | Symbol | Planck 2018 best fit | Varied? |
|-----------|--------|---------------------|----------|
| Hubble constant | $H_0$ | 67.4 km/s/Mpc | Yes |
| Total matter density | $\Omega_m$ | 0.315 | Yes |
| Dark energy density | $\Omega_\Lambda$ | 0.685 | Yes |

### The helper function

Study the function below. It takes $(H_0,\;\Omega_m,\;\Omega_\Lambda)$ and returns the theoretical $\mathcal{D}_\ell$ evaluated at the same multipoles as our data.

In [ ]:
def get_theory_Dl(H0, Omega_m, Omega_Lambda, ell_out=ell_data):
    '''
    Compute theoretical TT power spectrum D_ell using CAMB.

    Parameters
    ----------
    H0 : float             Hubble constant [km/s/Mpc]
    Omega_m : float        Total matter density parameter
    Omega_Lambda : float   Dark energy density parameter
    ell_out : array        Multipoles at which to return D_ell

    Returns
    -------
    Dl_theory : ndarray    D_ell values [muK^2]
    '''
    h = H0 / 100.0
    ombh2 = 0.0224                       # fixed
    omch2 = Omega_m * h**2 - ombh2       # CDM density

    # Guard against unphysical parameter combinations
    if omch2 < 0.001:
        return np.full_like(ell_out, 1e10, dtype=float)

    try:
        pars = camb.CAMBparams()
        pars.set_cosmology(
            H0=H0,
            ombh2=ombh2,
            omch2=omch2,
            omk=1.0 - Omega_m - Omega_Lambda,   # spatial curvature
            tau=0.054,
        )
        pars.InitPower.set_params(As=2.1e-9, ns=0.965)
        pars.set_for_lmax(int(max(ell_out)) + 100, lens_potential_accuracy=0)

        results = camb.get_results(pars)
        powers  = results.get_cmb_power_spectra(pars, CMB_unit='muK')
        totCL   = powers['total']            # shape (lmax+1, 4): TT, EE, BB, TE
        ells_camb = np.arange(totCL.shape[0])
        Dl_camb   = totCL[:, 0]              # TT spectrum (already D_ell)

        # Interpolate onto our data multipoles
        return np.interp(ell_out, ells_camb, Dl_camb)

    except Exception:
        return np.full_like(ell_out, 1e10, dtype=float)

Note that CAMB calls take much longer than I'd thought. So I created a pre-computed grid that you can use instead. You can do this through defining a new function ```get_theory_Dl_fast``` that can be used in place of ```get_theory_Dl``` everywhere that follows. To do the full calculations (not on colab) just switch which function you use.

In [ ]:
from scipy.interpolate import RegularGridInterpolator

data = np.load('camb_grid.npz')
interpolators = []
for m in range(len(data['ell'])):
    interp = RegularGridInterpolator(
        (data['H0'], data['Om'], data['OL']),
        data['Dl'][:, :, :, m],
        method='linear', bounds_error=False, fill_value=1e10
    )
    interpolators.append(interp)

def get_theory_Dl_fast(H0, Omega_m, Omega_Lambda):
    point = np.array([H0, Omega_m, Omega_Lambda])
    return np.array([interp(point)[0] for interp in interpolators])

Now we will explore how changing these three parameters shift the peaks in the power spectrum. We generate and overplot theoretical spectra for several parameter values on top of the data. We will first vary the Hubble constant between 60 and 75 km/s/Mpc, while keeping $\Omega_m = 0.31$ and $\Omega_\Lambda = 0.69$ fixed.

In [ ]:
plt.errorbar(ell_data, Dl_data, yerr=Dl_sigma, fmt='o', capsize=3,
             color='k', markersize=4, label='Planck data', zorder=5)

for H0_val in [60, 67, 75]:
    Dl_th = get_theory_Dl_fast(H0_val, 0.31, 0.69)
    plt.plot(ell_data, Dl_th, '-o', markersize=3,
             label=f'$H_0 = {H0_val}$')

plt.xscale('log')
plt.xlabel(r'$\ell$')
plt.ylabel(r'$\mathcal{D}_\ell\;[\mu\mathrm{K}^2]$')
plt.title(r'Effect of varying $H_0$')
plt.legend()
plt.tight_layout()
plt.show()

### Question

Explore how changing the value of $\Omega_m = 0.20, 0.31, 0.45$ changes the power spectrum by making a similar overplotted comparison. Keep $H_0 = 67.4$ and $\Omega_\Lambda = 0.69$. Which parameter mainly shifts the peaks left/right? Which mainly changes the peak heights? Why? (Think about how $H_0$ affects the angular diameter distance to the last scattering surface, and how gravity affects the acoustic oscillations.)

### Answer

In [ ]:
# YOUR CODE HERE

---
## Part 3 — $\chi^2$ Fitting with `scipy.optimize`

Before running a full MCMC, let us find a quick best fit using $\chi^2$ minimization.

The $\chi^2$ statistic is:

$$\chi^2(\theta) = \sum_i \frac{\left[\mathcal{D}_\ell^{\rm data}(i) - \mathcal{D}_\ell^{\rm theory}(i;\,\theta)\right]^2}{\sigma_i^2}$$

where $\theta = (H_0,\;\Omega_m,\;\Omega_\Lambda)$.

Write the $\chi^2$ function

In [ ]:
def chi_squared(params):
    '''
    Compute chi^2 between data and CAMB theory for given parameters.
    params : [H0, Omega_m, Omega_Lambda]
    '''
    H0, Omega_m, Omega_Lambda = params

    # YOUR CODE HERE — compute the theoretical D_ell and return chi^2


    return chi2

Let's do a quick test using the best-fit Planck values to see that you $\chi^2$ fit is working:


In [ ]:
# Quick test: what is chi^2 at the Planck best-fit values?
chi2_test = chi_squared([67.4, 0.315, 0.685])
print(f'chi^2 at Planck best fit = {chi2_test:.1f}')
print(f'Number of data points    = {len(ell_data)}')
print(f'Degrees of freedom       = {len(ell_data) - 3}')
print(f'Reduced chi^2            = {chi2_test / (len(ell_data) - 3):.2f}')

Now we can find the best fit with `scipy.optimize.minimize` to minimize $\chi^2$.

Start from an initial guess that is *not* the Planck best fit, for example $H_0 = 65$, $\Omega_m = 0.28$, $\Omega_\Lambda = 0.72$.

In [ ]:
initial_guess = [65.0, 0.28, 0.72]

result = minimize(chi_squared, initial_guess, options={'maxiter': 300})

H0_fit, Om_fit, OL_fit = result.x
print('=== scipy.optimize best fit ===')
print(f'H0             = {H0_fit:.2f} km/s/Mpc')
print(f'Omega_m        = {Om_fit:.4f}')
print(f'Omega_Lambda   = {OL_fit:.4f}')
print(f'Omega_k        = {1 - Om_fit - OL_fit:.4f}')
print(f'chi^2          = {result.fun:.1f}')
print(f'Reduced chi^2  = {result.fun / (len(ell_data) - 3):.2f}')

Finally, plot the best fit over the data, as you've done before.

In [ ]:
# YOUR CODE HERE — compute the theoretical D_ell and return chi^2


---
## Part 4 — Bayesian Parameter Estimation with `emcee`

The optimizer gives us a best fit, but not **uncertainties** or **degeneracies**. For that we need to sample the posterior distribution:

$$p(\theta \mid \text{data}) \propto \mathcal{L}(\text{data} \mid \theta)\;\pi(\theta)$$

where:
- $\mathcal{L} \propto e^{-\chi^2/2}$ is the likelihood (assuming Gaussian errors)
- $\pi(\theta)$ is the prior

Define the log-prior and log-posterior using **uniform (flat) priors** within physically reasonable ranges:

| Parameter | Prior range |
|-----------|------------|
| $H_0$ | [50, 90] km/s/Mpc |
| $\Omega_m$ | [0.05, 0.70] |
| $\Omega_\Lambda$ | [0.30, 0.95] |

In [ ]:
def log_prior(params):
    '''Flat prior: return 0 if in bounds, -inf otherwise.'''
    H0, Omega_m, Omega_Lambda = params

    if 50 < H0 < 90 and 0.05 < Omega_m < 0.70 and 0.30 < Omega_Lambda < 0.95:
        return 0.0
    return -np.inf


def log_likelihood(params):
    '''Gaussian log-likelihood = -chi^2 / 2.'''
    return -0.5 * chi_squared(params)


def log_posterior(params):
    '''Log-posterior = log-prior + log-likelihood.'''
    lp = log_prior(params)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(params)

# Quick sanity check
print(f'log_posterior at Planck best fit: {log_posterior([67.4, 0.315, 0.685]):.1f}')

Run the MCMC sampler. Initialize walkers in a small ball around our scipy best fit and let `emcee` explore the posterior. Each CAMB call takes ~0.1-0.5 s, so we keep the chain length modest. This is enough to see the posterior shape clearly.

In [ ]:
# MCMC configuration
ndim     = 3       # number of parameters
nwalkers = 16      # number of walkers (must be >= 2 * ndim)
nsteps   = 300     # number of MCMC steps per walker

# Initialize walkers in a small Gaussian ball around the scipy best fit
p0 = result.x + 1e-3 * np.random.randn(nwalkers, ndim) * np.array([1.0, 0.01, 0.01])

# Create the sampler and run!
sampler = emcee.EnsembleSampler(nwalkers, ndim, log_posterior)

print(f'Running MCMC: {nwalkers} walkers x {nsteps} steps = {nwalkers * nsteps} evaluations')
print('This may take a few minutes — each step calls CAMB...')

sampler.run_mcmc(p0, nsteps, progress=True)

print('Done!')

Inspect the chains (trace plots) to check that the walkers have converged. Plot each parameter vs. step number for all walkers.

In [ ]:
labels = [r'$H_0$', r'$\Omega_m$', r'$\Omega_\Lambda$']
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)

samples = sampler.get_chain()  # shape: (nsteps, nwalkers, ndim)

for i in range(ndim):
    ax = axes[i]
    ax.plot(samples[:, :, i], alpha=0.3, linewidth=0.5)
    ax.set_ylabel(labels[i])
    ax.axhline(result.x[i], color='r', linestyle='--', linewidth=1)

axes[-1].set_xlabel('Step number')
axes[0].set_title('MCMC Trace Plots — Check for Convergence')
plt.tight_layout()
plt.show()

### Question

Look at your trace plots. Do the walkers appear to have "settled down" after some initial burn-in period? Choose a burn-in value (e.g., the first 100 steps) to discard.

### Answer

Set the burn-in below and use the code to generate a corner plot.

In [ ]:
# Discard burn-in and flatten the chains
burn_in =

flat_samples = sampler.get_chain(discard=burn_in, flat=True)
print(f'Posterior samples shape: {flat_samples.shape}')

# ---- Summary statistics ----
print('\n=== Posterior Parameter Estimates ===')
for i, name in enumerate(labels):
    q = np.percentile(flat_samples[:, i], [16, 50, 84])
    med = q[1]
    lo  = q[1] - q[0]
    hi  = q[2] - q[1]
    print(f'  {name:20s} = {med:.3f}  (+{hi:.3f} / -{lo:.3f})')

In [ ]:
# Corner plot — shows 1D and 2D marginal posteriors
fig = corner.corner(
    flat_samples,
    labels=labels,
    quantiles=[0.16, 0.5, 0.84],
    show_titles=True,
    title_fmt='.3f',
    truths=[67.4, 0.315, 0.685],       # Planck 2018 published values
    truth_color='crimson',
)
fig.suptitle('Posterior Distributions (red lines = Planck 2018 values)', y=1.02)
plt.show()

---
## Part 5 — Plot your best MCMC fit

Now repeat the plot you made in Part 2, but with your best fit model from MCMC.

In [ ]:
# Use the posterior medians as 'best' estimates
H0_mcmc  = np.median(flat_samples[:, 0])
Om_mcmc  = np.median(flat_samples[:, 1])
OL_mcmc  = np.median(flat_samples[:, 2])

Dl_mcmc = get_theory_Dl_fast(H0_mcmc, Om_mcmc, OL_mcmc)

# YOUR CODE HERE to plot your best MCMC fit

Compare your results to the Planck results by filling in the table below with your results:

| Parameter | Your MCMC estimate | Planck 2018 published |
|-----------|-------------------|----------------------|
| $H_0$ [km/s/Mpc] |  | $67.4 \pm 0.5$ |
| $\Omega_m$ |  | $0.315 \pm 0.007$ |
| $\Omega_\Lambda$ |  | $0.685 \pm 0.007$ |
| $\Omega_k = 1 - \Omega_m - \Omega_\Lambda$ |  | $\sim 0$ (consistent with flat) |

### Final Questions

1. **Are your results consistent with the Planck published values?** Are they within your error bars?

2. **Is the universe flat?** What does your constraint on $\Omega_k = 1 - \Omega_m - \Omega_\Lambda$ tell you?

3. **Degeneracies:** Look at your corner plot. Are any parameters correlated with each other? Why might $H_0$ and $\Omega_m$ be (anti-)correlated?

4. **Limitations:** We fixed $\Omega_b h^2$, $n_s$, $\tau$, and $A_s$. If we freed those too, what would happen to our uncertainties? Would we need more data?

5. **Hubble tension:** The local distance-ladder measurement gives $H_0 \approx 73 \pm 1$ km/s/Mpc. Is your CMB-based $H_0$ consistent with that?